In [1]:
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from pathlib import Path

print("Librerías cargadas correctamente")

/Users/juana/Desktop/tfm-data-science-gtm/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Librerías cargadas correctamente


In [2]:
# Overpass API endpoint
overpass_url = "https://overpass-api.de/api/interpreter"

# Bounding box del distrito Centro de Madrid (España)
# Formato Overpass: (sur, oeste, norte, este)
south = 40.4050
west = -3.7275
north = 40.4290
east = -3.6900

# Query con bbox global — sin nombres, así que no hay confusión con otros Madrid del mundo
query = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["amenity"="restaurant"];
  node["amenity"="bar"];
  node["amenity"="cafe"];
  node["amenity"="pub"];
  node["amenity"="fast_food"];
);
out center;
"""

print("Query preparada — Centro Madrid España")
print(f"Bounding box: sur={south}, oeste={west}, norte={north}, este={east}")

Query preparada — Centro Madrid España
Bounding box: sur=40.405, oeste=-3.7275, norte=40.429, este=-3.69


In [3]:
print("Descargando datos de OpenStreetMap... (puede tardar 30-60 segundos)")

# Añadimos headers con User-Agent para que Overpass acepte la petición
headers = {
    "User-Agent": "TFM-DataScience-JuanAlonso/1.0 (proyecto académico Máster Data Science)"
}

response = requests.post(
    overpass_url,
    data={"data": query},
    headers=headers
)

print(f"Status code: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"✅ Respuesta recibida. Número de elementos: {len(data['elements'])}")
else:
    print("❌ Error en la petición. Contenido devuelto:")
    print(response.text[:500])

Descargando datos de OpenStreetMap... (puede tardar 30-60 segundos)
Status code: 200
✅ Respuesta recibida. Número de elementos: 3036


In [4]:
print(f"Status code: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print("---- Primeras 500 caracteres de la respuesta ----")
print(response.text[:500])

Status code: 200
Content-Type: application/json
---- Primeras 500 caracteres de la respuesta ----
{
  "version": 0.6,
  "generator": "Overpass API 0.7.62.11 87bfad18",
  "osm3s": {
    "timestamp_osm_base": "2026-09-14T21:40:56Z",
    "copyright": "The data included in this document is from www.openstreetmap.org. The data is made available under ODbL."
  },
  "elements": [

{
  "type": "node",
  "id": 26065697,
  "lat": 40.4287093,
  "lon": -3.7019720,
  "tags": {
    "addr:city": "Madrid",
    "addr:housenumber": "7",
    "addr:postcode": "28004",
    "addr:street": "Glorieta de Bilbao",
  


In [5]:
# Ver los primeros elementos para entender qué ha devuelto
print(f"Total elementos: {len(data['elements'])}\n")

# Cuántos son 'node' y cuántos son 'way'
tipos = {}
for elem in data['elements']:
    t = elem.get('type', 'desconocido')
    tipos[t] = tipos.get(t, 0) + 1

print(f"Tipos: {tipos}\n")

# Ver los primeros 3 elementos completos
for i, elem in enumerate(data['elements'][:3]):
    print(f"--- Elemento {i+1} ---")
    print(elem)
    print()
    

Total elementos: 3036

Tipos: {'node': 3036}

--- Elemento 1 ---
{'type': 'node', 'id': 26065697, 'lat': 40.4287093, 'lon': -3.701972, 'tags': {'addr:city': 'Madrid', 'addr:housenumber': '7', 'addr:postcode': '28004', 'addr:street': 'Glorieta de Bilbao', 'amenity': 'restaurant', 'email': 'info@cafecomercialmadrid.com', 'name': 'Café Comercial', 'opening_hours': 'Mo-Th 08:30-01:00; Fr-Su 08:30-02:00', 'phone': '+34 910 88 25 25', 'website': 'https://cafecomercialmadrid.com/', 'wikidata': 'Q5017237', 'wikimedia_commons': 'Category:Café Comercial', 'wikipedia': 'es:Café Comercial'}}

--- Elemento 2 ---
{'type': 'node', 'id': 26065699, 'lat': 40.4270276, 'lon': -3.7016997, 'tags': {'addr:city': 'Madrid', 'addr:housenumber': '95', 'addr:postcode': '28004', 'addr:street': 'Calle de Fuencarral', 'amenity': 'restaurant', 'branch': 'Fuencarral', 'brand': 'Honest Greens', 'brand:wikidata': 'Q116859710', 'cuisine': 'organic', 'diet:gluten_free': 'yes', 'diet:healthy': 'yes', 'diet:vegan': 'yes', 

In [6]:
# Verificación de que estamos en Madrid España
if data['elements']:
    primer_elem = data['elements'][0]
    lat = primer_elem.get('lat')
    lon = primer_elem.get('lon')
    print(f"Primer elemento — lat: {lat}, lon: {lon}")
    
    # Madrid España está aprox en lat 40.4, lon -3.7
    if lat and 40.3 < lat < 40.5 and -3.8 < lon < -3.6:
        print("✅ Coordenadas correctas: Madrid España")
    else:
        print(f"⚠️ Coordenadas no coinciden con Madrid España")

# Además, un vistazo a los tipos de establecimiento
tipos_amenity = {}
for elem in data['elements']:
    tags = elem.get('tags', {})
    amenity = tags.get('amenity', 'sin_amenity')
    tipos_amenity[amenity] = tipos_amenity.get(amenity, 0) + 1

print("\n--- Distribución por tipo ---")
for tipo, count in sorted(tipos_amenity.items(), key=lambda x: -x[1]):
    print(f"  {tipo}: {count}")
    

Primer elemento — lat: 40.4287093, lon: -3.701972
✅ Coordenadas correctas: Madrid España

--- Distribución por tipo ---
  restaurant: 1688
  bar: 527
  cafe: 368
  fast_food: 249
  pub: 204


In [7]:
# =====================================
# DESCARGA FARMACIAS EN CENTRO MADRID
# =====================================

query_farmacias = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["amenity"="pharmacy"];
  way["amenity"="pharmacy"];
);
out center;
"""

print("Descargando farmacias...")
response_farmacias = requests.post(
    overpass_url,
    data={"data": query_farmacias},
    headers=headers
)

if response_farmacias.status_code == 200:
    data_farmacias = response_farmacias.json()
    print(f"✅ Farmacias encontradas: {len(data_farmacias['elements'])}")
else:
    print(f"❌ Error: {response_farmacias.status_code}")
    print(response_farmacias.text[:300])

Descargando farmacias...
✅ Farmacias encontradas: 149


In [8]:
# =====================================
# DESCARGA CLÍNICAS DENTALES EN CENTRO MADRID
# =====================================

query_dental = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["amenity"="dentist"];
  node["healthcare"="dentist"];
  way["amenity"="dentist"];
  way["healthcare"="dentist"];
);
out center;
"""

print("Descargando clínicas dentales...")
response_dental = requests.post(
    overpass_url,
    data={"data": query_dental},
    headers=headers
)

if response_dental.status_code == 200:
    data_dental = response_dental.json()
    print(f"✅ Clínicas dentales encontradas: {len(data_dental['elements'])}")
else:
    print(f"❌ Error: {response_dental.status_code}")
    print(response_dental.text[:300])

Descargando clínicas dentales...
✅ Clínicas dentales encontradas: 40


In [9]:
# =====================================
# ANÁLISIS COMPARATIVO DE COBERTURA
# =====================================

def analizar_cobertura(data, nombre_sector):
    """Analiza qué porcentaje de registros tiene cada campo clave."""
    total = len(data['elements'])
    if total == 0:
        return None
    
    # Contadores
    con_nombre = 0
    con_direccion = 0
    con_cp = 0
    con_telefono = 0
    con_web = 0
    con_email = 0
    con_horario = 0
    
    for elem in data['elements']:
        tags = elem.get('tags', {})
        
        if tags.get('name'):
            con_nombre += 1
        if tags.get('addr:street'):
            con_direccion += 1
        if tags.get('addr:postcode'):
            con_cp += 1
        if tags.get('phone') or tags.get('contact:phone'):
            con_telefono += 1
        if tags.get('website') or tags.get('contact:website'):
            con_web += 1
        if tags.get('email') or tags.get('contact:email'):
            con_email += 1
        if tags.get('opening_hours'):
            con_horario += 1
    
    return {
        'sector': nombre_sector,
        'total_registros': total,
        'nombre_%': round(con_nombre / total * 100, 1),
        'direccion_%': round(con_direccion / total * 100, 1),
        'cp_%': round(con_cp / total * 100, 1),
        'telefono_%': round(con_telefono / total * 100, 1),
        'web_%': round(con_web / total * 100, 1),
        'email_%': round(con_email / total * 100, 1),
        'horario_%': round(con_horario / total * 100, 1),
    }

# Analizar los 3 sectores
resultados = []
resultados.append(analizar_cobertura(data, "Hostelería"))
resultados.append(analizar_cobertura(data_farmacias, "Farmacias"))
resultados.append(analizar_cobertura(data_dental, "Clínicas dentales"))

# Mostrar como DataFrame para verlo bonito
df_cobertura = pd.DataFrame(resultados)
print("=" * 80)
print("COMPARATIVA DE COBERTURA POR SECTOR")
print("=" * 80)
df_cobertura

COMPARATIVA DE COBERTURA POR SECTOR


,sector,total_registros,nombre_%,direccion_%,cp_%,telefono_%,web_%,email_%,horario_%
0,Hostelería,3036,98.1,75.1,64.6,36.6,33.7,6.2,16.9
1,Farmacias,149,47.7,93.3,90.6,91.3,1.3,0.0,16.1
2,Clínicas dentales,40,92.5,62.5,47.5,45.0,50.0,7.5,10.0


In [10]:
# =====================================
# DESCARGA GIMNASIOS EN CENTRO MADRID
# =====================================

query_gym = f"""
[out:json][timeout:90][bbox:{south},{west},{north},{east}];
(
  node["leisure"="fitness_centre"];
  node["leisure"="sports_centre"];
  node["sport"="fitness"];
  way["leisure"="fitness_centre"];
  way["leisure"="sports_centre"];
  way["sport"="fitness"];
);
out center;
"""

print("Descargando gimnasios y centros deportivos...")
response_gym = requests.post(
    overpass_url,
    data={"data": query_gym},
    headers=headers
)

if response_gym.status_code == 200:
    data_gym = response_gym.json()
    print(f"✅ Gimnasios/centros deportivos encontrados: {len(data_gym['elements'])}")
    
    # Analizar cobertura
    resultado_gym = analizar_cobertura(data_gym, "Gimnasios")
    print(f"\nCobertura de campos:")
    for campo, valor in resultado_gym.items():
        print(f"  {campo}: {valor}")
else:
    print(f"❌ Error: {response_gym.status_code}")
    print(response_gym.text[:300])

Descargando gimnasios y centros deportivos...
✅ Gimnasios/centros deportivos encontrados: 51

Cobertura de campos:
  sector: Gimnasios
  total_registros: 51
  nombre_%: 84.3
  direccion_%: 58.8
  cp_%: 43.1
  telefono_%: 23.5
  web_%: 43.1
  email_%: 3.9
  horario_%: 7.8


In [11]:
# =====================================
# COMPARATIVA: RESTAURANTES vs BARES vs OTROS
# =====================================

# Separar por amenity
restaurantes = [e for e in data['elements'] if e.get('tags', {}).get('amenity') == 'restaurant']
bares = [e for e in data['elements'] if e.get('tags', {}).get('amenity') == 'bar']
cafes = [e for e in data['elements'] if e.get('tags', {}).get('amenity') == 'cafe']

def cobertura_lista(elementos, nombre):
    total = len(elementos)
    if total == 0:
        return None
    con_nombre = sum(1 for e in elementos if e.get('tags', {}).get('name'))
    con_dir = sum(1 for e in elementos if e.get('tags', {}).get('addr:street'))
    con_cp = sum(1 for e in elementos if e.get('tags', {}).get('addr:postcode'))
    con_tel = sum(1 for e in elementos if e.get('tags', {}).get('phone') or e.get('tags', {}).get('contact:phone'))
    con_web = sum(1 for e in elementos if e.get('tags', {}).get('website') or e.get('tags', {}).get('contact:website'))
    con_cocina = sum(1 for e in elementos if e.get('tags', {}).get('cuisine'))
    
    return {
        'tipo': nombre,
        'total': total,
        'nombre_%': round(con_nombre / total * 100, 1),
        'direccion_%': round(con_dir / total * 100, 1),
        'cp_%': round(con_cp / total * 100, 1),
        'telefono_%': round(con_tel / total * 100, 1),
        'web_%': round(con_web / total * 100, 1),
        'cocina_%': round(con_cocina / total * 100, 1),
    }

resultados = [
    cobertura_lista(restaurantes, "Restaurantes"),
    cobertura_lista(bares, "Bares"),
    cobertura_lista(cafes, "Cafeterías"),
]

df_comp = pd.DataFrame(resultados)
print("=" * 80)
print("COBERTURA POR SUBSECTOR DE HOSTELERÍA")
print("=" * 80)
df_comp

COBERTURA POR SUBSECTOR DE HOSTELERÍA


,tipo,total,nombre_%,direccion_%,cp_%,telefono_%,web_%,cocina_%
0,Restaurantes,1688,98.6,78.8,69.1,45.6,41.9,54.1
1,Bares,527,98.1,79.7,68.7,26.6,18.0,5.3
2,Cafeterías,368,96.7,68.5,52.4,23.6,29.1,25.5


In [12]:
# =====================================
# CONVERTIR RESTAURANTES A DATAFRAME
# =====================================

registros_rest = []

for elem in restaurantes:
    tags = elem.get('tags', {})
    
    registro = {
        'osm_id': elem.get('id'),
        'osm_type': elem.get('type'),
        'lat': elem.get('lat'),
        'lon': elem.get('lon'),
        'name': tags.get('name'),
        'cuisine': tags.get('cuisine'),
        'addr_street': tags.get('addr:street'),
        'addr_housenumber': tags.get('addr:housenumber'),
        'addr_postcode': tags.get('addr:postcode'),
        'addr_city': tags.get('addr:city'),
        'phone': tags.get('phone') or tags.get('contact:phone'),
        'website': tags.get('website') or tags.get('contact:website'),
        'opening_hours': tags.get('opening_hours'),
        'outdoor_seating': tags.get('outdoor_seating'),
        'takeaway': tags.get('takeaway'),
        'delivery': tags.get('delivery'),
        'wheelchair': tags.get('wheelchair'),
    }
    
    registros_rest.append(registro)

df_rest = pd.DataFrame(registros_rest)
print(f"DataFrame de restaurantes creado: {len(df_rest)} filas × {len(df_rest.columns)} columnas")
print(f"\nPrimeras 3 filas:")
df_rest.head(3)

DataFrame de restaurantes creado: 1688 filas × 17 columnas

Primeras 3 filas:


,osm_id,osm_type,lat,lon,name,cuisine,addr_street,addr_housenumber,addr_postcode,addr_city,phone,website,opening_hours,outdoor_seating,takeaway,delivery,wheelchair
0,26065697,node,40.428709,-3.701972,Café Comercial,None,Glorieta de Bilbao,7,28004,Madrid,+34 910 88 25 25,https://cafecomercialmadrid.com/,Mo-Th 08:30-01:00; Fr-Su 08:30-02:00,None,None,None,None
1,26065699,node,40.427028,-3.701700,Honest Greens,organic,Calle de Fuencarral,95,28004,Madrid,None,https://honestgreens.com/,"Mo-Th 08:30-23:00, Fr 08:30-24:00, Sa 09:30-24...",None,None,None,no
2,26808568,node,40.425766,-3.712090,La Parrilla de Nino,None,Plaza de Cristino Martos,2,28015,Madrid,+34 915 59 60 11,None,None,yes,None,None,limited


In [13]:
!pip install pyarrow

In [14]:
!pip install --upgrade pyarrow

In [15]:
# =====================================
# GUARDAR CAPA RAW
# =====================================

from pathlib import Path

# Asegurar que la carpeta existe
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Guardar en Parquet (formato eficiente para Data Science)
ruta_parquet = raw_dir / "restaurantes_centro_madrid_osm.parquet"
df_rest.to_parquet(ruta_parquet, index=False)

# Guardar también como CSV para poder abrirlo fácil en Excel/inspección
ruta_csv = raw_dir / "restaurantes_centro_madrid_osm.csv"
df_rest.to_csv(ruta_csv, index=False)

print(f"✅ Guardado como Parquet: {ruta_parquet}")
print(f"✅ Guardado como CSV: {ruta_csv}")
print(f"\nRegistros guardados: {len(df_rest)}")
print(f"Tamaño CSV: {ruta_csv.stat().st_size / 1024:.1f} KB")
print(f"Tamaño Parquet: {ruta_parquet.stat().st_size / 1024:.1f} KB")

✅ Guardado como Parquet: ../data/raw/restaurantes_centro_madrid_osm.parquet
✅ Guardado como CSV: ../data/raw/restaurantes_centro_madrid_osm.csv

Registros guardados: 1688
Tamaño CSV: 197.7 KB
Tamaño Parquet: 110.7 KB


In [16]:
# =====================================
# CARGAR CENSO DE LOCALES DE MADRID
# =====================================

# Ruta al CSV que has descargado
# Necesito que lo muevas a data/raw/ del proyecto primero
# Después ajustamos ruta

import pandas as pd

# IMPORTANTE: primero mueve el fichero descargado a la carpeta data/raw/ del proyecto
# El nombre lo puedes dejar tal cual, con el número del principio
ruta_censo = "../data/raw/200085-5-censo-locales.csv"

# El CSV usa separador ";" y encoding UTF-8 con BOM
df_censo = pd.read_csv(
    ruta_censo,
    sep=';',
    encoding='utf-8-sig',  # 'utf-8-sig' quita el BOM del principio
    low_memory=False,
)

print(f"✅ Censo cargado: {len(df_censo)} registros")
print(f"Columnas: {len(df_censo.columns)}")
print(f"\nPrimeras columnas: {list(df_censo.columns)[:15]}")

✅ Censo cargado: 225628 registros
Columnas: 47

Primeras columnas: ['id_local', 'id_distrito_local', 'desc_distrito_local', 'id_barrio_local', 'desc_barrio_local', 'cod_barrio_local', 'id_seccion_censal_local', 'desc_seccion_censal_local', 'coordenada_x_local', 'coordenada_y_local', 'id_tipo_acceso_local', 'desc_tipo_acceso_local', 'id_situacion_local', 'desc_situacion_local', 'id_vial_edificio']


In [17]:
# =====================================
# EXPLORAR CATEGORÍAS DEL CENSO
# =====================================

# Ver las columnas relevantes para categorizar
print("=== Distribución por SECCIÓN ===")
print(df_censo['desc_seccion'].value_counts().head(10))

print("\n=== Distribución por DIVISIÓN (top 15) ===")
print(df_censo['desc_division'].value_counts().head(15))

print("\n=== Distribución por SITUACIÓN ===")
print(df_censo['desc_situacion_local'].value_counts())

print("\n=== Distribución por DISTRITO (top 10) ===")
print(df_censo['desc_distrito_local'].value_counts().head(10))

=== Distribución por SECCIÓN ===
desc_seccion
COMERCIO AL POR MAYOR Y AL POR MENOR; REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS    53597
HOSTELERÍA                                                                               30881
OTROS SERVICIOS                                                                          17200
ACTIVIDADES ADMINISTRATIVAS Y SERVICIOS AUXLIARES                                        11176
ACTIVIDADES PROFESIONALES, CIENTÍFICAS Y TÉCNICAS                                        10431
ACTIVIDADES SANITARIAS Y DE SERVICIOS SOCIALES                                            9189
EDUCACIÓN                                                                                 6712
TRANSPORTE Y ALMACENAMIENTO                                                               4711
SIN ACTIVIDAD                                                                             4439
INDUSTRIA MANUFACTURERA                                                                   4396
Name

In [18]:
# =====================================
# FILTRAR RESTAURANTES DEL CENTRO ABIERTOS
# =====================================

# Filtro triple:
# 1. Distrito CENTRO
# 2. División = Servicios de comidas y bebidas
# 3. Situación = Abierto
df_rest_censo = df_censo[
    (df_censo['desc_distrito_local'].str.strip() == 'CENTRO') &
    (df_censo['desc_division'] == 'SERVICIOS DE COMIDAS Y BEBIDAS') &
    (df_censo['desc_situacion_local'] == 'Abierto')
].copy()

print(f"✅ Restaurantes/bares/cafés del Centro y abiertos: {len(df_rest_censo)}")

# Ver qué epígrafes concretos hay
print("\n=== Epígrafes específicos ===")
print(df_rest_censo['desc_epigrafe'].value_counts())

✅ Restaurantes/bares/cafés del Centro y abiertos: 3254

=== Epígrafes específicos ===
desc_epigrafe
BAR RESTAURANTE                                                                           810
RESTAURANTE                                                                               739
CAFETERIA                                                                                 494
BAR CON COCINA                                                                            362
BAR SIN COCINA                                                                            261
BAR ESPECIAL SIN ACTUACIONES                                                              218
RESTAURANTES DE COMIDA RAPIDA                                                              77
TABERNA                                                                                    66
COMERCIO AL POR MENOR DE VINOS Y ALCOHOLES (BODEGA) CON CONSUMO                            42
SERVICIOS DE COMEDOR EN CENTROS EDUCATIVOS Y CENTROS D

In [19]:
# =====================================
# FILTRAR SOLO RESTAURANTES DEL CENSO
# (para comparar con nuestros restaurantes de OSM)
# =====================================

# Nos quedamos con los epígrafes que son restaurantes propiamente
epigrafes_restaurantes = [
    'RESTAURANTE',
    'BAR RESTAURANTE',
    'RESTAURANTES DE COMIDA RAPIDA',
]

df_rest_censo_solo = df_rest_censo[
    df_rest_censo['desc_epigrafe'].isin(epigrafes_restaurantes)
].copy()

print(f"✅ Restaurantes en el censo (Centro, abiertos): {len(df_rest_censo_solo)}")
print(f"   - RESTAURANTE: {(df_rest_censo_solo['desc_epigrafe'] == 'RESTAURANTE').sum()}")
print(f"   - BAR RESTAURANTE: {(df_rest_censo_solo['desc_epigrafe'] == 'BAR RESTAURANTE').sum()}")
print(f"   - RESTAURANTES DE COMIDA RAPIDA: {(df_rest_censo_solo['desc_epigrafe'] == 'RESTAURANTES DE COMIDA RAPIDA').sum()}")

# Comparación
print(f"\n=== COMPARATIVA CENTRO MADRID ===")
print(f"Restaurantes en OSM:                  1.688")
print(f"Restaurantes en censo Ayuntamiento:   {len(df_rest_censo_solo)}")
print(f"Diferencia:                            {len(df_rest_censo_solo) - 1688}")

✅ Restaurantes en el censo (Centro, abiertos): 1626
   - RESTAURANTE: 739
   - BAR RESTAURANTE: 810
   - RESTAURANTES DE COMIDA RAPIDA: 77

=== COMPARATIVA CENTRO MADRID ===
Restaurantes en OSM:                  1.688
Restaurantes en censo Ayuntamiento:   1626
Diferencia:                            -62


In [20]:
# =====================================
# PREPARAR EL CRUCE POR DIRECCIÓN
# =====================================

# Preparar df_rest (OSM) — normalizar dirección
def normalizar_texto(s):
    if pd.isna(s):
        return None
    return str(s).strip().upper()

df_rest['calle_norm'] = df_rest['addr_street'].apply(normalizar_texto)
df_rest['numero_norm'] = df_rest['addr_housenumber'].apply(normalizar_texto)

# Preparar df_rest_censo_solo (Ayuntamiento) — normalizar dirección
# El censo tiene calle en 'desc_vial_edificio' y número en 'num_edificio'
df_rest_censo_solo['calle_norm'] = df_rest_censo_solo['desc_vial_edificio'].apply(normalizar_texto)
# Los números vienen como "000005" — hay que limpiarlos
df_rest_censo_solo['numero_norm'] = df_rest_censo_solo['num_edificio'].apply(
    lambda x: str(int(x)) if pd.notna(x) and str(x).strip().isdigit() else None
)

# Ver muestras para comparar
print("=== MUESTRAS DE OSM ===")
print(df_rest[['name', 'calle_norm', 'numero_norm']].head(10))

print("\n=== MUESTRAS DE CENSO ===")
print(df_rest_censo_solo[['rotulo', 'calle_norm', 'numero_norm']].head(10))

=== MUESTRAS DE OSM ===
                  name                        calle_norm numero_norm
0       Café Comercial                GLORIETA DE BILBAO           7
1        Honest Greens               CALLE DE FUENCARRAL          95
2  La Parrilla de Nino          PLAZA DE CRISTINO MARTOS           2
3            Taj Mahal          CALLE DEL DUQUE DE OSUNA           6
4          La Pomarada             CALLE DEL CONDE DUQUE        None
5                Sexto              CALLE DE FERNANDO VI           6
6         Brasa y Leña  PLAZA DEL COMANDANTE LAS MORENAS           3
7          Picado Fino                              None        None
8           El Viajero                PLAZA DE LA CEBADA          11
9            La Malaje                              None        None

=== MUESTRAS DE CENSO ===
                     rotulo        calle_norm numero_norm
0                    VITACA           BARCELO        None
1               ZAATAR & CO           ACUERDO        None
5               

In [21]:
# =====================================
# NORMALIZACIÓN DE DIRECCIONES (v2)
# =====================================

import re

def normalizar_calle(s):
    """Normaliza el nombre de la calle: quita prefijos y limpia."""
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    # Quitar prefijos comunes de OSM
    prefijos = [
        'CALLE DE LA ', 'CALLE DE LOS ', 'CALLE DE LAS ',
        'CALLE DEL ', 'CALLE DE ', 'CALLE ',
        'PLAZA DE LA ', 'PLAZA DEL ', 'PLAZA DE ', 'PLAZA ',
        'GLORIETA DE LA ', 'GLORIETA DEL ', 'GLORIETA DE ', 'GLORIETA ',
        'PASEO DEL ', 'PASEO DE LA ', 'PASEO DE ', 'PASEO ',
        'AVENIDA DE ', 'AVENIDA ',
        'RONDA DE ', 'RONDA ',
        'TRAVESIA DE ', 'TRAVESIA ',
        'CARRERA DE ', 'CARRERA ',
        'CUESTA DE ', 'CUESTA ',
        'BAJADA DE ', 'BAJADA ',
    ]
    for pref in prefijos:
        if s.startswith(pref):
            s = s[len(pref):]
            break
    # Eliminar múltiples espacios
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def normalizar_numero(s):
    """Extrae el número puro de una cadena tipo '000005' → '5' o '5A' → '5'."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    # Sacar solo la parte numérica del principio
    m = re.match(r'^0*(\d+)', s)
    if m:
        return m.group(1)
    return None

# Aplicar a OSM
df_rest['calle_norm'] = df_rest['addr_street'].apply(normalizar_calle)
df_rest['numero_norm'] = df_rest['addr_housenumber'].apply(normalizar_numero)

# Aplicar al censo
df_rest_censo_solo['calle_norm'] = df_rest_censo_solo['desc_vial_edificio'].apply(normalizar_calle)
df_rest_censo_solo['numero_norm'] = df_rest_censo_solo['num_edificio'].apply(normalizar_numero)

# Ver muestras nuevas
print("=== MUESTRAS DE OSM (v2) ===")
print(df_rest[['name', 'calle_norm', 'numero_norm']].head(10))

print("\n=== MUESTRAS DE CENSO (v2) ===")
print(df_rest_censo_solo[['rotulo', 'calle_norm', 'numero_norm']].head(10))

=== MUESTRAS DE OSM (v2) ===
                  name              calle_norm numero_norm
0       Café Comercial                  BILBAO           7
1        Honest Greens              FUENCARRAL          95
2  La Parrilla de Nino         CRISTINO MARTOS           2
3            Taj Mahal          DUQUE DE OSUNA           6
4          La Pomarada             CONDE DUQUE        None
5                Sexto             FERNANDO VI           6
6         Brasa y Leña  COMANDANTE LAS MORENAS           3
7          Picado Fino                    None        None
8           El Viajero                  CEBADA          11
9            La Malaje                    None        None

=== MUESTRAS DE CENSO (v2) ===
                     rotulo        calle_norm numero_norm
0                    VITACA           BARCELO           5
1               ZAATAR & CO           ACUERDO          31
5                      MUNE         SAN MATEO          17
7            LA DESCUBIERTA         BARCELONA          12


In [22]:
# =====================================
# CRUCE OSM ↔ CENSO POR DIRECCIÓN
# =====================================

# Preparar la clave de cruce: calle + número
df_rest['clave_cruce'] = df_rest['calle_norm'] + '|' + df_rest['numero_norm']
df_rest_censo_solo['clave_cruce'] = df_rest_censo_solo['calle_norm'] + '|' + df_rest_censo_solo['numero_norm']

# Cuántos restaurantes de OSM tienen dirección completa
osm_con_direccion = df_rest[df_rest['clave_cruce'].notna() & 
                             ~df_rest['clave_cruce'].str.contains('None', na=True)].copy()
print(f"Restaurantes OSM con dirección completa: {len(osm_con_direccion)} / {len(df_rest)}")
print(f"({len(osm_con_direccion)/len(df_rest)*100:.1f}%)")

# Cuántos del censo tienen dirección completa
censo_con_direccion = df_rest_censo_solo[df_rest_censo_solo['clave_cruce'].notna() & 
                                          ~df_rest_censo_solo['clave_cruce'].str.contains('None', na=True)].copy()
print(f"\nRestaurantes censo con dirección completa: {len(censo_con_direccion)} / {len(df_rest_censo_solo)}")
print(f"({len(censo_con_direccion)/len(df_rest_censo_solo)*100:.1f}%)")

# Hacer el merge
df_cruce = osm_con_direccion.merge(
    censo_con_direccion,
    on='clave_cruce',
    how='left',
    suffixes=('_osm', '_censo')
)

# Contar cuántos se enriquecieron
enriquecidos = df_cruce['id_local'].notna().sum()
print(f"\n=== RESULTADO DEL CRUCE ===")
print(f"Restaurantes OSM procesados:    {len(osm_con_direccion)}")
print(f"Restaurantes enriquecidos:      {enriquecidos}")
print(f"Restaurantes sin match:         {len(osm_con_direccion) - enriquecidos}")
print(f"Tasa de éxito del cruce:        {enriquecidos/len(osm_con_direccion)*100:.1f}%")

Restaurantes OSM con dirección completa: 1259 / 1688
(74.6%)

Restaurantes censo con dirección completa: 1626 / 1626
(100.0%)

=== RESULTADO DEL CRUCE ===
Restaurantes OSM procesados:    1259
Restaurantes enriquecidos:      818
Restaurantes sin match:         441
Tasa de éxito del cruce:        65.0%


In [23]:
# =====================================
# QUÉ VARIABLES APORTA EL CRUCE
# =====================================

# Filtrar solo los enriquecidos
df_enriquecido = df_cruce[df_cruce['id_local'].notna()].copy()

print(f"Total de restaurantes enriquecidos: {len(df_enriquecido)}\n")

# Muestra de cómo queda un restaurante enriquecido
print("=== EJEMPLO DE UN RESTAURANTE ENRIQUECIDO ===")
ejemplo = df_enriquecido.iloc[0]
print(f"\nNombre OSM:              {ejemplo['name']}")
print(f"Rótulo Ayuntamiento:     {ejemplo['rotulo']}")
print(f"Dirección:               {ejemplo['calle_norm_osm']} {ejemplo['numero_norm_osm']}")
print(f"Distrito:                {ejemplo['desc_distrito_local']}")
print(f"Barrio:                  {ejemplo['desc_barrio_local']}")
print(f"Sección censal:          {ejemplo['desc_seccion_censal_local']}")
print(f"Epígrafe oficial:        {ejemplo['desc_epigrafe']}")
print(f"Tipo de acceso:          {ejemplo['desc_tipo_acceso_local']}")
print(f"Cocina OSM:              {ejemplo['cuisine']}")
print(f"Web OSM:                 {ejemplo['website']}")

# Distribución de nuevas variables ganadas
print("\n\n=== BARRIOS CUBIERTOS (top 10) ===")
print(df_enriquecido['desc_barrio_local'].value_counts().head(10))

print("\n=== EPÍGRAFES ENRIQUECIDOS ===")
print(df_enriquecido['desc_epigrafe'].value_counts())

Total de restaurantes enriquecidos: 818

=== EJEMPLO DE UN RESTAURANTE ENRIQUECIDO ===

Nombre OSM:              Café Comercial
Rótulo Ayuntamiento:     S/R
Dirección:               BILBAO 7
Distrito:                CENTRO              
Barrio:                  JUSTICIA            
Sección censal:          89.0
Epígrafe oficial:        RESTAURANTES DE COMIDA RAPIDA
Tipo de acceso:          Puerta Calle
Cocina OSM:              None
Web OSM:                 https://cafecomercialmadrid.com/


=== BARRIOS CUBIERTOS (top 10) ===
desc_barrio_local
SOL                     195
UNIVERSIDAD             190
JUSTICIA                148
PALACIO                 133
CORTES                   95
EMBAJADORES              57
Name: count, dtype: int64

=== EPÍGRAFES ENRIQUECIDOS ===
desc_epigrafe
RESTAURANTE                      400
BAR RESTAURANTE                  391
RESTAURANTES DE COMIDA RAPIDA     27
Name: count, dtype: int64


In [24]:
# =====================================
# CONVERTIR COORDENADAS UTM DEL CENSO A LAT/LON
# =====================================

from pyproj import Transformer

# Las coordenadas UTM del censo están en ETRS89 / UTM zona 30N (EPSG:25830)
# Las de OSM están en WGS84 lat/lon (EPSG:4326)
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)

# Aplicar la conversión al censo
def convertir_utm(fila):
    x, y = fila['coordenada_x_local'], fila['coordenada_y_local']
    if pd.notna(x) and pd.notna(y):
        lon, lat = transformer.transform(x, y)
        return pd.Series({'lat_censo': lat, 'lon_censo': lon})
    return pd.Series({'lat_censo': None, 'lon_censo': None})

df_rest_censo_solo[['lat_censo', 'lon_censo']] = df_rest_censo_solo.apply(convertir_utm, axis=1)

# Verificar que las coordenadas son correctas (deben estar en Madrid)
print("=== VERIFICACIÓN DE COORDENADAS ===")
print(df_rest_censo_solo[['rotulo', 'lat_censo', 'lon_censo']].head(5))

print(f"\nRango de latitudes censo: {df_rest_censo_solo['lat_censo'].min():.4f} a {df_rest_censo_solo['lat_censo'].max():.4f}")
print(f"Rango de longitudes censo: {df_rest_censo_solo['lon_censo'].min():.4f} a {df_rest_censo_solo['lon_censo'].max():.4f}")
print(f"(Deberían ser aprox lat 40.4X y lon -3.7X para Madrid)")

=== VERIFICACIÓN DE COORDENADAS ===
           rotulo  lat_censo  lon_censo
0          VITACA  40.426558  -3.700788
1     ZAATAR & CO  40.428794  -3.707991
5            MUNE  40.424816  -3.696986
7  LA DESCUBIERTA  40.415187  -3.702627
9  GRACIAS, PADRE  40.422246  -3.697242

Rango de latitudes censo: 0.0000 a 40.4304
Rango de longitudes censo: -7.4887 a -3.6914
(Deberían ser aprox lat 40.4X y lon -3.7X para Madrid)


In [25]:
# =====================================
# FILTRAR COORDENADAS VÁLIDAS Y ANALIZAR SIN MATCH
# =====================================

# Filtro: solo coordenadas dentro de Madrid ciudad
df_censo_geo = df_rest_censo_solo[
    (df_rest_censo_solo['lat_censo'].between(40.35, 40.50)) &
    (df_rest_censo_solo['lon_censo'].between(-3.80, -3.60))
].copy()

print(f"Censo con coordenadas válidas: {len(df_censo_geo)} / {len(df_rest_censo_solo)}")

# Restaurantes OSM que NO cruzaron por dirección
osm_sin_match = df_cruce[df_cruce['id_local'].isna()].copy()
print(f"\nRestaurantes OSM sin match por dirección: {len(osm_sin_match)}")

# Restaurantes OSM que ni tenían dirección
osm_sin_direccion = df_rest[df_rest['clave_cruce'].str.contains('None', na=True)].copy()
print(f"Restaurantes OSM sin dirección completa: {len(osm_sin_direccion)}")

Censo con coordenadas válidas: 1535 / 1626

Restaurantes OSM sin match por dirección: 639
Restaurantes OSM sin dirección completa: 429


In [29]:
# =====================================
# CRUCE POR PROXIMIDAD GEOGRÁFICA (v2)
# =====================================

import numpy as np
from scipy.spatial import cKDTree

# Restaurantes OSM que YA se enriquecieron (ya cruzaron por dirección)
ids_enriquecidos = df_enriquecido['osm_id'].tolist()

# Restaurantes OSM pendientes (todos los que NO están enriquecidos)
osm_pendientes = df_rest[~df_rest['osm_id'].isin(ids_enriquecidos)].copy()

print(f"Restaurantes OSM pendientes de enriquecer: {len(osm_pendientes)}")

# Filtrar solo los que tienen coordenadas
osm_pendientes = osm_pendientes.dropna(subset=['lat', 'lon']).copy()
print(f"De ellos, con coordenadas válidas: {len(osm_pendientes)}")

# Construir árbol de búsqueda con las coordenadas del censo
coords_censo = df_censo_geo[['lat_censo', 'lon_censo']].values
tree = cKDTree(coords_censo)

# Para cada restaurante OSM pendiente, buscar el censo más cercano
coords_osm = osm_pendientes[['lat', 'lon']].values
distancias, indices = tree.query(coords_osm, k=1)

# Convertir distancia de grados a metros
distancias_metros = distancias * 111320

# Añadir la info al DataFrame
osm_pendientes['distancia_metros'] = distancias_metros
osm_pendientes['indice_censo'] = indices

# Contar cuántos están a menos de X metros
print("\n=== ANÁLISIS DE PROXIMIDAD ===")
for umbral in [10, 20, 30, 50, 100]:
    n = (distancias_metros < umbral).sum()
    print(f"OSM con censo a < {umbral}m: {n} ({n/len(osm_pendientes)*100:.1f}%)")

Restaurantes OSM pendientes de enriquecer: 1068
De ellos, con coordenadas válidas: 1068

=== ANÁLISIS DE PROXIMIDAD ===
OSM con censo a < 10m: 465 (43.5%)
OSM con censo a < 20m: 640 (59.9%)
OSM con censo a < 30m: 737 (69.0%)
OSM con censo a < 50m: 813 (76.1%)
OSM con censo a < 100m: 879 (82.3%)


In [28]:
# Ver qué columnas tenemos en df_enriquecido
print("Columnas de df_enriquecido que contienen 'osm_id':")
print([c for c in df_enriquecido.columns if 'osm_id' in c.lower()])

print("\nColumnas de df_enriquecido que contienen 'id':")
print([c for c in df_enriquecido.columns if 'id' in c.lower()])

Columnas de df_enriquecido que contienen 'osm_id':
['osm_id']

Columnas de df_enriquecido que contienen 'id':
['osm_id', 'id_local', 'id_distrito_local', 'id_barrio_local', 'id_seccion_censal_local', 'id_tipo_acceso_local', 'id_situacion_local', 'id_vial_edificio', 'id_ndp_edificio', 'id_clase_ndp_edificio', 'id_vial_acceso', 'id_ndp_acceso', 'id_clase_ndp_acceso', 'id_agrupacion', 'id_tipo_agrup', 'id_planta_agrupado', 'id_local_agrupado', 'id_seccion', 'id_division', 'id_epigrafe']


In [30]:
# =====================================
# CONSOLIDAR EL ENRIQUECIMIENTO FINAL
# =====================================

# Umbral elegido: 30 metros (mejor equilibrio precisión/recall)
UMBRAL_METROS = 30

# Filtrar los OSM pendientes que sí matchean por proximidad
matches_proximidad = osm_pendientes[osm_pendientes['distancia_metros'] < UMBRAL_METROS].copy()

# Añadir la info del censo correspondiente
info_censo = df_censo_geo.reset_index(drop=True)
matches_proximidad['desc_barrio_local'] = matches_proximidad['indice_censo'].map(info_censo['desc_barrio_local'])
matches_proximidad['desc_distrito_local'] = matches_proximidad['indice_censo'].map(info_censo['desc_distrito_local'])
matches_proximidad['desc_epigrafe'] = matches_proximidad['indice_censo'].map(info_censo['desc_epigrafe'])
matches_proximidad['id_seccion_censal_local'] = matches_proximidad['indice_censo'].map(info_censo['id_seccion_censal_local'])
matches_proximidad['metodo_match'] = 'proximidad'

# Marcar los que ya estaban enriquecidos como método 'direccion'
df_enriquecido = df_enriquecido.copy()
df_enriquecido['metodo_match'] = 'direccion'

# RESUMEN FINAL
print("=" * 60)
print("RESUMEN FINAL DEL ENRIQUECIMIENTO")
print("=" * 60)

total_osm = len(df_rest)
por_direccion = len(df_enriquecido)
por_proximidad = len(matches_proximidad)
total_enriquecidos = por_direccion + por_proximidad
sin_enriquecer = total_osm - total_enriquecidos

print(f"\nTotal restaurantes OSM:                   {total_osm}")
print(f"Enriquecidos por cruce de dirección:      {por_direccion} ({por_direccion/total_osm*100:.1f}%)")
print(f"Enriquecidos por proximidad (<{UMBRAL_METROS}m):        {por_proximidad} ({por_proximidad/total_osm*100:.1f}%)")
print(f"TOTAL ENRIQUECIDOS:                       {total_enriquecidos} ({total_enriquecidos/total_osm*100:.1f}%)")
print(f"Sin enriquecer:                           {sin_enriquecer} ({sin_enriquecer/total_osm*100:.1f}%)")

RESUMEN FINAL DEL ENRIQUECIMIENTO

Total restaurantes OSM:                   1688
Enriquecidos por cruce de dirección:      818 (48.5%)
Enriquecidos por proximidad (<30m):        737 (43.7%)
TOTAL ENRIQUECIDOS:                       1555 (92.1%)
Sin enriquecer:                           133 (7.9%)


In [31]:
# =====================================
# GUARDAR CAPA PROCESSED
# =====================================

from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Guardar los enriquecidos por dirección
df_enriquecido.to_parquet(processed_dir / "restaurantes_enriquecidos_direccion.parquet", index=False)
df_enriquecido.to_csv(processed_dir / "restaurantes_enriquecidos_direccion.csv", index=False)

# Guardar los enriquecidos por proximidad
matches_proximidad.to_parquet(processed_dir / "restaurantes_enriquecidos_proximidad.parquet", index=False)
matches_proximidad.to_csv(processed_dir / "restaurantes_enriquecidos_proximidad.csv", index=False)

# Guardar el censo filtrado también en processed
df_rest_censo_solo.to_parquet(processed_dir / "censo_restaurantes_madrid.parquet", index=False)
df_rest_censo_solo.to_csv(processed_dir / "censo_restaurantes_madrid.csv", index=False)

print("✅ Ficheros guardados en data/processed/:")
print(f"   - restaurantes_enriquecidos_direccion.parquet ({len(df_enriquecido)} filas)")
print(f"   - restaurantes_enriquecidos_proximidad.parquet ({len(matches_proximidad)} filas)")
print(f"   - censo_restaurantes_madrid.parquet ({len(df_rest_censo_solo)} filas)")

✅ Ficheros guardados en data/processed/:
   - restaurantes_enriquecidos_direccion.parquet (818 filas)
   - restaurantes_enriquecidos_proximidad.parquet (737 filas)
   - censo_restaurantes_madrid.parquet (1626 filas)
